In [5]:
import pandas as pd
import joblib
import os
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. ENTER YOUR DATA HERE (THE ARRAY)
# Format: ("URL", Flag) -> 0 for Safe, 1 for Phishing
# ==========================================
feedback_array = [
   

]


# ==========================================
# 2. FEATURE EXTRACTION LOGIC
# ==========================================
try:
    model = joblib.load('csec_rf_model.pkl')
    expected_features = list(model.feature_names_in_)
except FileNotFoundError:
    print("Error: csec_rf_model.pkl not found. Please train your model first.")
    expected_features = []

def get_counts(text, is_missing=False):
    if is_missing:
        return {char: -1 for char in ['dot', 'hyphen', 'underline', 'slash', 'questionmark', 'equal', 'at', 'and', 'exclamation', 'space', 'tilde', 'comma', 'plus', 'asterisk', 'hashtag', 'dollar', 'percent']}
    return {
        'dot': text.count('.'), 'hyphen': text.count('-'), 'underline': text.count('_'),
        'slash': text.count('/'), 'questionmark': text.count('?'), 'equal': text.count('='),
        'at': text.count('@'), 'and': text.count('&'), 'exclamation': text.count('!'),
        'space': text.count(' '), 'tilde': text.count('~'), 'comma': text.count(','),
        'plus': text.count('+'), 'asterisk': text.count('*'), 'hashtag': text.count('#'),
        'dollar': text.count('$'), 'percent': text.count('%')
    }

def extract_111_features(url):
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url
    
    parsed = urlparse(url)
    domain = parsed.netloc
    path = parsed.path
    params = parsed.query
    
    has_directory = '/' in path and len(path) > 1
    has_file = '.' in os.path.basename(path)
    has_params = len(params) > 0
    
    directory = os.path.dirname(path) if has_directory else ""
    file = os.path.basename(path) if has_file else ""
    
    u_counts = get_counts(url)
    d_counts = get_counts(domain)
    dir_counts = get_counts(directory, not has_directory)
    f_counts = get_counts(file, not has_file)
    p_counts = get_counts(params, not has_params)
    
    features = {}
    
    for char in u_counts:
        features[f'qty_{char}_url'] = u_counts[char]
        features[f'qty_{char}_domain'] = d_counts[char]
        features[f'qty_{char}_directory'] = dir_counts[char]
        features[f'qty_{char}_file'] = f_counts[char]
        features[f'qty_{char}_params'] = p_counts[char]
        
    features['length_url'] = len(url)
    features['domain_length'] = len(domain)
    features['directory_length'] = len(directory) if has_directory else -1
    features['file_length'] = len(file) if has_file else -1
    features['params_length'] = len(params) if has_params else -1
    features['qty_params'] = len(params.split('&')) if has_params else -1
    features['email_in_url'] = 1 if re.search(r'[\w\.-]+@[\w\.-]+', url) else 0
    features['tls_ssl_certificate'] = 1 if url.startswith('https') else 0
    
    # Fill remaining columns with -1
    features_dict = {col: features.get(col, -1) for col in expected_features}
    return features_dict


# ==========================================
# 3. AUTOMATED BATCH PROCESSING & RETRAINING
# ==========================================
print("=======================================")
print(f" PROCESSING {len(feedback_array)} URLs FROM ARRAY")
print("=======================================")

new_data_list = []

for url_input, flag_input in feedback_array:
    print(f"Extracting features for: {url_input} (Flag: {flag_input})")
    row_data = extract_111_features(url_input)
    row_data['phishing'] = flag_input
    new_data_list.append(row_data)

if len(new_data_list) > 0:
    df_new = pd.DataFrame(new_data_list)
    
    print("\n[1] Updating dataset_small.csv...")
    try:
        # Load the original dataset
        df_original = pd.read_csv('dataset_small.csv')
        
        # Merge the old data with the new data
        df_combined = pd.concat([df_original, df_new], ignore_index=True)
        
        # Save it back to dataset_small.csv (Permanent Update)
        df_combined.to_csv('dataset_small.csv', index=False)
        print("✅ dataset_small.csv updated successfully!")
        
        print("\n[2] Retraining the AI Model...")
        
        # THE FIX: Drop any rows where the answer (phishing column) is blank (NaN)
        df_combined = df_combined.dropna(subset=['phishing'])
        
        # Now separate the features (X) and the answers (y)
        X = df_combined.drop('phishing', axis=1)
        
        # It is now 100% safe to force the answers to be integers
        y = df_combined['phishing'].astype(int) 
        
        # Ensure all X data is numeric too, filling any blank features with -1
        X = X.apply(pd.to_numeric, errors='coerce').fillna(-1)
        
        new_rf = RandomForestClassifier(n_estimators=100, random_state=42)
        new_rf.fit(X, y)
        
        # Overwrite the model
        joblib.dump(new_rf, 'csec_rf_model.pkl')
        print("✅ SUCCESS! The model (csec_rf_model.pkl) has been updated with your new data.")
        
    except Exception as e:
        print(f"❌ Error during update: {e}")
else:
    print("No data in array. Nothing to update.")

 PROCESSING 0 URLs FROM ARRAY
No data in array. Nothing to update.
